In [0]:
dbutils.fs.rm("/FileStore/GENEVA/DATABRICKS_UNIFIED/table",true)

In [0]:
%sql
DROP TABLE IF EXISTS default.GVA_GL_ACTIVITY;
DROP TABLE IF EXISTS default.GVA_GL_ACTIVITY_AUDIT;
DROP TABLE IF EXISTS default.STG_GVA_GL_ACTIVITY;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS default.STG_GVA_GL_ACTIVITY(
CUST_ACCOUNT_CHAINID STRING,
PORTFOLIO_CHAINID STRING,
STRATEGY_CHAINID STRING,
ACCOUNTING_CALENDAR_ID STRING,
ACCOUNTING_PERIOD_NAME STRING,
RPT_PRIOR_KNOWLEDGE_DATE STRING,
RPT_KNOWLEDGE_DATE STRING,
RPT_PERIOD_END_DATE STRING,
RPT_PERIOD_START_DATE STRING,
CURRENCY_CHAINID STRING,
DEBIT_CREDIT_INDICATOR STRING,
ACCOUNTING_DATE STRING,
FINANCIAL_ACCOUNT_CHAINID STRING,
LEDGER_AMOUNT_BOOK STRING,
LEDGER_AMOUNT_LOCAL STRING,
RUNNING_BALANCE_BOOK STRING,
RUNNING_BALANCE_LOCAL STRING,
INVESTMENT_CHAINID STRING,
GVA_ASSETTYPE STRING,
GVA_INVESTMENTTYPE STRING,
TRAN_ID STRING,
TRANSACTION_TYPE STRING,
TRADE_DATE STRING,
SETTLE_DATE STRING,
GVA_SECT STRING,
GVA_CAT STRING,
GVA_SUBCATEGORY STRING,
GVA_DETAIL STRING,
GVA_DETAIL_CUSTOM STRING,
GVA_INVENTORYSTATE STRING,
RUN_TYPE STRING,
IS_FINAL STRING,
CLIENT_SECURITY_ID STRING,
FX_RATE_DERIVED STRING,
FX_INVESTMENT_DESC STRING,
TRAN_KNOWLEDGE_END_DATE STRING,
TRAN_KNOWLEDGE_BEGIN_DATE STRING,
TRAN_CHANGE STRING,
ACCT_CHANGE STRING,
ACCT_ACTION STRING,
TRAN_DETAIL STRING,
TRAN_COMMENTS STRING,
UNIQUE_TRAN_ID STRING,
CLIENT_SEQ STRING,
USER_ID STRING,
LONG_SHORT STRING,
TAXLOT_ID STRING,
PORTFOLIO_CODE STRING,
CLIENT_CODE STRING,
CREDIT_FACILITY_CHAINID STRING,
STRATEGY_INDICATOR STRING,
EFFECTIVE_DATE DATE,
CREATETIME TIMESTAMP,
FILENAME STRING,
SOURCE_NAME STRING)
USING delta 
LOCATION "/FileStore/GENEVA/DATABRICKS_UNIFIED/table/STG_GVA_GL_ACTIVITY";

CREATE TABLE IF NOT EXISTS default.GVA_GL_ACTIVITY(
CUST_ACCOUNT_CHAINID VARCHAR(100) NOT NULL,
PORTFOLIO_CHAINID VARCHAR(100) NOT NULL,
STRATEGY_CHAINID VARCHAR(100),
ACCOUNTING_CALENDAR_ID VARCHAR(255) NOT NULL,
ACCOUNTING_PERIOD_NAME VARCHAR(255),
RPT_PRIOR_KNOWLEDGE_DATE TIMESTAMP,
RPT_KNOWLEDGE_DATE TIMESTAMP,
RPT_PERIOD_END_DATE TIMESTAMP NOT NULL,
RPT_PERIOD_START_DATE TIMESTAMP,
CURRENCY_CHAINID VARCHAR(255),
DEBIT_CREDIT_INDICATOR VARCHAR(255) NOT NULL,
ACCOUNTING_DATE TIMESTAMP,
FINANCIAL_ACCOUNT_CHAINID VARCHAR(100) NOT NULL,
LEDGER_AMOUNT_BOOK DECIMAL(38,18),
LEDGER_AMOUNT_LOCAL DECIMAL(38,18),
RUNNING_BALANCE_BOOK DECIMAL(38,18),
RUNNING_BALANCE_LOCAL DECIMAL(38,18),
INVESTMENT_CHAINID VARCHAR(100) NOT NULL,
GVA_ASSETTYPE VARCHAR(255),
GVA_INVESTMENTTYPE VARCHAR(255),
TRAN_ID VARCHAR(255),
TRANSACTION_TYPE VARCHAR(255),
TRADE_DATE TIMESTAMP,
SETTLE_DATE TIMESTAMP,
GVA_SECT VARCHAR(255) NOT NULL,
GVA_CAT VARCHAR(255),
GVA_SUBCATEGORY VARCHAR(255),
GVA_DETAIL VARCHAR(255),
GVA_DETAIL_CUSTOM VARCHAR(255),
GVA_INVENTORYSTATE VARCHAR(255) NOT NULL,
RUN_TYPE VARCHAR(255) NOT NULL,
IS_FINAL VARCHAR(10),
CLIENT_SECURITY_ID VARCHAR(256),
FX_RATE_DERIVED DECIMAL(38,18),
FX_INVESTMENT_DESC VARCHAR(255),
TRAN_KNOWLEDGE_END_DATE TIMESTAMP,
TRAN_KNOWLEDGE_BEGIN_DATE TIMESTAMP,
TRAN_CHANGE VARCHAR(255),
ACCT_CHANGE VARCHAR(3999),
ACCT_ACTION VARCHAR(3999),
TRAN_DETAIL VARCHAR(255),
TRAN_COMMENTS VARCHAR(3999),
UNIQUE_TRAN_ID VARCHAR(255),
CLIENT_SEQ VARCHAR(256),
USER_ID VARCHAR(255),
LONG_SHORT VARCHAR(10) NOT NULL,
TAXLOT_ID VARCHAR(100) NOT NULL,
PORTFOLIO_CODE VARCHAR(100),
CLIENT_CODE VARCHAR(100),
CREDIT_FACILITY_CHAINID VARCHAR(32),
STRATEGY_INDICATOR VARCHAR(10) NOT NULL,
EFFECTIVE_DATE DATE,
ODS_LOAD_DATETIME TIMESTAMP NOT NULL,
FILENAME STRING,
SOURCE_NAME STRING,
PRODUCER_LOAD_DATETIME TIMESTAMP NOT NULL)
USING delta 
LOCATION "/FileStore/GENEVA/DATABRICKS_UNIFIED/table/GVA_GL_ACTIVITY"
TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS default.GVA_GL_ACTIVITY_AUDIT (
  FILENAME STRING,
  RUN_ID BIGINT,
  STATUS STRING,
  PROCESSED_AT TIMESTAMP,
  PROCESSED_DATE DATE
)
USING DELTA
PARTITIONED BY (PROCESSED_DATE)
LOCATION "/FileStore/GENEVA/DATABRICKS_UNIFIED/table/GVA_GL_ACTIVITY_AUDIT";

In [0]:
import org.apache.spark.sql._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import scala.util.Try
import org.apache.spark.sql.Dataset
import java.text.SimpleDateFormat
import java.util.Date
import org.apache.spark.sql.streaming.Trigger
import scala.collection.JavaConverters._

spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")

case class Task(
  taskName: String,
  inputFormat: Option[String] = None,
  outputFormat: Option[String] = None,
  schema: Option[String] = None,
  options: Option[Map[String, String]] = None,
  sourceToTargetMapping: Option[Map[String,String]] = None,
  rules: Option[Seq[ColRule]] = None
)

case class FeedConfig(
  feedName: String,
  tasks: Seq[Task]
)

case class ColRule(
  name: String,
  dataType: DataType,
  nullable: Boolean,
  varcharMaxLen: Option[Int] = None,
  tsPattern: Option[String] = None,
  datePattern: Option[String] = None,
  decPrecision: Option[Int] = None,
  decScale: Option[Int] = None
)

def stringToDataType(t:String, prec: Option[Int], scale: Option[Int]): DataType = t.toUpperCase match {
  case "STRING" => StringType
  case "DECIMAL" =>
    (prec, scale) match {
      case(Some(p),Some(s)) => DecimalType(p,s)
      case(Some(p),None) => DecimalType(p,0)
      case _ => DecimalType(38,18)
    }
  case "TIMESTAMP" => TimestampType
  case "DATE" => DateType
  case "INT" => IntegerType
  case "DOUBLE" => DoubleType
  case "BOOLEAN" => BooleanType
  case _ => StringType
}

In [0]:
def loadFeedConfig(configPath: String): FeedConfig = {
    import spark.implicits._

    val configDF = spark.read.option("multiline",true).json(configPath)
    val feedName = Try(configDF.select($"feedName").first().getString(0)).getOrElse("UnknownFeed")
    val taskRows = configDF.select(explode($"tasks").as("task")).select("task.*").collect()

    val tasks =  taskRows.toSeq.map { row =>
      val taskName = Try(row.getAs[String]("taskName")).getOrElse("UnknownTask")
      val inputFormat = Try(row.getAs[String]("inputFormat")).toOption
      val outputFormat = Try(row.getAs[String]("outputFormat")).toOption
      val schema = Try(row.getAs[String]("schema")).toOption

      val optionsRow = row.getAs[Row]("options")
      val options = if (optionsRow != null) {
        Some(optionsRow.schema.fieldNames.map { key =>
          key -> optionsRow.getAs[String](key)
        }.toMap)
      } else None

      val mapping = if (row.schema.fieldNames.contains("sourceToTargetMapping") && row.getAs[Row]("sourceToTargetMapping") != null) {
        Some(row.getAs[Row]("sourceToTargetMapping").schema.fieldNames.map { key =>
          key -> optionsRow.getAs[String](key)
        }.toMap)
      } else None

      val rulesDF = Try(row.getAs[Seq[Row]]("rules")).getOrElse(Seq.empty)
      val rules = if(rulesDF != null && rulesDF.nonEmpty) {
        Some(rulesDF.map { r =>
          val name = r.getAs[String]("name")
          val dataType = r.getAs[String]("type")
          val nullable = Try(r.getAs[Boolean]("nullable")).toOption.getOrElse(true)
          val varcharMaxLen = Try(r.getAs[Long]("varcharMaxLen")).toOption.map(_.toInt)
          val tsPattern = Try(r.getAs[String]("tsPattern")).toOption
          val datePattern = Try(r.getAs[String]("datePattern")).toOption
          val decPrecision = Try(r.getAs[Long]("decPrecision")).toOption.map(_.toInt)
          val decScale = Try(r.getAs[Long]("decScale")).toOption.map(_.toInt)

          ColRule(
            name,
            stringToDataType(dataType, decPrecision, decScale),
            nullable,
            varcharMaxLen, tsPattern, datePattern, decPrecision, decScale
          )
        })
      } else None
      Task(taskName,inputFormat,outputFormat,schema,options,mapping,rules)
    }
    FeedConfig(feedName,tasks)
}


In [0]:
def addValidationColumns(df:DataFrame, rules: Seq[ColRule]): DataFrame = {

  val validations = rules.map { r =>
    val colName = r.name
    val baseCheck: Column = r.dataType match {
      case TimestampType =>
        col(colName).isNull || to_timestamp(col(colName), r.tsPattern.getOrElse("M/d/yyyy HH:mm:ss")).isNotNull
      case DateType =>
        col(colName).isNull || to_date(col(colName), r.datePattern.getOrElse("M/d/yyyy")).isNotNull
      case StringType =>
        val maxLen = r.varcharMaxLen.getOrElse(Int.MaxValue)
        col(colName).isNull || length(col(colName))<=maxLen
      case d: DecimalType =>
        val precision = r.decPrecision.getOrElse(38)
        val scale = r.decScale.getOrElse(18)
        val decimalRegex = s"""^-?\\d{1,${precision - scale}}(\\.\\d{1,${scale}})?$$"""
        col(colName).isNull || (col(colName).cast(DecimalType(precision, scale)).isNotNull && regexp_replace(col(colName), ",", "").rlike(decimalRegex))
      case _ =>lit(true)
    }

    val finalCheck = 
      if (!r.nullable)
        baseCheck && col(colName).isNotNull
      else
        baseCheck
    
    (colName,finalCheck)
  }

  val invalidColsExpr = array(validations.map { case (colName, expr) =>
        when(!expr, lit(colName))
      }.toSeq: _*)

  df.withColumn("invalidColumns", concat_ws(",", invalidColsExpr))
    .withColumn("isValidRow", size(filter(invalidColsExpr, c => c.isNotNull && c =!= "")) === 0)

}

In [0]:
def castForBase(validDF: DataFrame, rules: Seq[ColRule]): DataFrame = {
  val castedCols: Seq[Column] = rules.map { r =>
    r.dataType match {
      case StringType     => col(r.name).cast(StringType).as(r.name)
      case d: DecimalType => expr(s"cast(${r.name} as DECIMAL(${d.precision},${d.scale}))").as(r.name)
      case TimestampType  => to_timestamp(col(r.name),r.tsPattern.getOrElse("M/d/yyyy HH:mm:ss")).as(r.name)
      case DateType       => to_date(col(r.name),r.datePattern.getOrElse("M/d/yyyy")).as(r.name)
      case other          => col(r.name).cast(other).as(r.name)
    }
  }

  val auditCols = Seq(col("STRATEGY_INDICATOR"),
                      col("EFFECTIVE_DATE"),
                      col("CREATETIME").as("ODS_LOAD_DATETIME"),
                      col("FILENAME"),
                      col("SOURCE_NAME"),
                      to_timestamp(regexp_extract(col("FILENAME"), "_(\\d{17})\\.csv$",1),"yyyyMMddHHmmssSSS").as("PRODUCER_LOAD_DATETIME")
                      )
  validDF.select((castedCols ++ auditCols): _*)
}

In [0]:
def cloudFileExists(filePath: String,filePattern: String): Boolean = {
  val files = dbutils.fs.ls(filePath)
  val fileExists = files.exists { fileInfo =>
      val fileName = fileInfo.name
          fileName.contains(filePattern)
      }
  fileExists
}

def generateNackMessageFile(nackMessageDF: Dataset[_], fileCompletePath: String, fileCompleteName: String): Unit = {
  val filePath = fileCompletePath
  val fileName = fileCompleteName
  val TStampFolders = new SimpleDateFormat("yyyyMMdd").format(new Date())
  val (year, month, date) = (TStampFolders.slice(0,4), TStampFolders.slice(4,6) , TStampFolders.slice(6,8))
  val fileLocation = filePath + "/" + year + "/" + month + "/" + date + "/"
  val TStamp = new SimpleDateFormat("yyyyMMddHHmmss").format(new Date())
  val file = fileLocation + fileName + "*.csv"
  val PartFiles = fileLocation + "part-00000*.csv"
  if(nackMessageDF.count()>0)
  {
    dbutils.fs.mkdirs(fileLocation)

    //Export the Data to CSV
    nackMessageDF.repartition(1)
      .write.mode("append")
      .format("csv")
      .option("timestampFormat", "yyyy/MM/dd HH:mm:ss ZZ")
      .option("header",true)
      .save(fileLocation)

	  if (cloudFileExists(fileLocation,s"part-00000")) 
	  {
      println("File Exists : Replacing Files!")
      dbutils.fs.mv(dbutils.fs.ls(fileLocation).filter(file => file.name.endsWith("csv")).filter(file => file.name.startsWith("part-00000"))(0).path, fileLocation + "/" + fileName + "_" + TStamp + ".csv")
      println("Removing Auto Generated Files!")
      dbutils.fs.rm(dbutils.fs.ls(fileLocation).filter(file => file.name.startsWith("_committed")).filter(file => file.name.startsWith("_"))(0).path, true)
      dbutils.fs.rm(dbutils.fs.ls(fileLocation).filter(file => file.name.startsWith("_started")).filter(file => file.name.startsWith("_"))(0).path, true)
      dbutils.fs.rm(dbutils.fs.ls(fileLocation).filter(file => file.name.startsWith("_SUCCESS")).filter(file => file.name.startsWith("_"))(0).path, true)
    }
    else 
	  {
      println(s"No File Exists: ${PartFiles}")
    }
  }
}

In [0]:
val configPath = "dbfs:/FileStore/GENEVA/DATABRICKS_UNIFIED/resources/gva_gl_activity_feed_config.json"
val cfg = loadFeedConfig(configPath).tasks(0)
val options = cfg.options.getOrElse(Map.empty)
val schema:String = cfg.schema.getOrElse("").replace("\\\"", "\"")
val customSchema = DataType.fromJson(schema).asInstanceOf[StructType]
val SourceType = "Fact"
val strategyIndicator = if(SourceType=="Fact") "STRATEGY_CHAINID" else "STRATEGY_CHAINID_DIM"
val effectiveDate = if(SourceType=="Fact") "RPT_PERIOD_END_DATE" else "RPT_PERIOD_END_DATE_DIM"

In [0]:
val streamDF = spark.readStream.format("cloudFiles")
              .option("cloudFiles.format", "csv")
              .option("maxFilesPerTrigger",100)
              .option("header", "true")
              .option("rescuedDataColumn", "_rescued_data")
              .schema(customSchema)
              .load(options.getOrElse("sourcePath",""))
              .withColumn("STRATEGY_INDICATOR",when(col(strategyIndicator).isNotNull,"Y").otherwise("N"))
              .withColumn("EFFECTIVE_DATE",to_date(to_timestamp(col(effectiveDate), "MM/dd/yyyy HH:mm:ss")))
              .withColumn("CREATETIME", current_timestamp())
              .withColumn("FILENAME", element_at(reverse(split(input_file_name(),"/")),1))
              .withColumn("SOURCE_NAME", lit("GENEVA"))

In [0]:
val query = streamDF
              .writeStream
              .option("maxFilesPerTrigger","100")
              .trigger(Trigger.AvailableNow())
              .queryName("GVA_GL_ACTIVITY")
              .option("checkpointLocation",options.getOrElse("checkpointLocation",""))
              .foreachBatch { (batchDF: DataFrame, batchId: Long) =>

                val today = current_date()
                
                //RAW Table Processing - append-only
                batchDF
                  .write
                  .option("mergeSchema",true)
                  .mode("append")
                  .format("delta")
                  .saveAsTable(options.getOrElse("rawTable",""))

                //Validate current micro-batch only
                val validatedDF = addValidationColumns(batchDF,cfg.rules.getOrElse(Seq.empty)).cache()
                
                val fileHealth = validatedDF.groupBy("filename")
                                    .agg(count(when(col("isValidRow") === lit(false),1)).as("invalidCount"))
                                    .withColumn("fileValid", col("invalidCount") === 0)
                
                val validFilesDF = fileHealth.filter(col("fileValid")).select("filename").distinct()
                val invalidFilesDF = fileHealth.filter(!col("fileValid")).select("filename").distinct()
                
                //New Files - which are not processed
                val processedDF = spark.table(options.getOrElse("auditTable",""))
                                    .filter(col("PROCESSED_DATE") === today)
                                    .select("filename")
                                    .distinct()

                val newValidFilesDF = validFilesDF.join(processedDF, Seq("filename"), "left_anti")
                val newInvalidFilesDF = invalidFilesDF.join(processedDF, Seq("filename"), "left_anti")
                
                //Nack File Generation and Audit Table Update
                val newlyInvalidRows = validatedDF.filter(!(col("isValidRow"))).join(broadcast(newInvalidFilesDF), Seq("filename"))
                if(!newlyInvalidRows.rdd.isEmpty)
                {
                  generateNackMessageFile(newlyInvalidRows, options.getOrElse("rejectCSVPath",""), (options.getOrElse("baseTable","")).split("\\.")(1))
                  newInvalidFilesDF
                    .select("filename")
                    .withColumn("run_id",lit(batchId))
                    .withColumn("status", lit("REJECTED"))
                    .withColumn("processed_at",current_timestamp())
                    .withColumn("processed_date",today)
                    .write.format("delta").mode("append").saveAsTable(options.getOrElse("auditTable",""))
                }
                
                //Base Table Ingestion and Audit Table Update
                val newlyValidRows = validatedDF.join(broadcast(newValidFilesDF), Seq("filename"))
                if(!newlyValidRows.rdd.isEmpty)
                {
                val baseOut = castForBase(newlyValidRows, cfg.rules.getOrElse(Seq.empty))
                baseOut.drop("_rescued_data").write.format("delta").mode("append").saveAsTable(options.getOrElse("baseTable",""))
                newValidFilesDF
                    .select("filename")
                    .withColumn("run_id",lit(batchId))
                    .withColumn("status", lit("SUCCESS"))
                    .withColumn("processed_at",current_timestamp())
                    .withColumn("processed_date",today)
                    .write.format("delta").mode("append").saveAsTable(options.getOrElse("auditTable",""))
                }
                
                validatedDF.unpersist()
                ()
              }
              .start()

query.awaitTermination()

In [0]:
%sql
SELECT * FROM default.raw_GVA_GL_ACTIVITY;

In [0]:
%sql
SELECT * FROM default.gva_gl_activity;

In [0]:
%sql
select * from default.gva_gl_activity_audit;

In [0]:
%sql
DESCRIBE HISTORY default.gva_gl_activity;

In [0]:
%python
for i in dbutils.secrets.list("aei-adl-secrets"):
    print(i.key)

In [0]:
import org.apache.spark.sql.functions._

val df = Seq("example_file_20240329123456789.csv").toDF("filename")
val resultDF = df.filter(col("filename").rlike(".*_\\d{17}\\.csv$"))